# S7-01: 워크플로 패턴 — Chaining, Routing, Parallelization
**Skilljar L02-L04: Parallelization / Chaining / Routing Workflows**

## 학습 목표
- Chaining 워크플로로 순차적 LLM 파이프라인을 구축한다
- Routing 워크플로로 입력 분류 기반 전문 처리기를 구현한다
- Parallelization (Sectioning, Voting)으로 동시 처리를 구현한다
- 건축공학 구조 검토에 워크플로 패턴을 적용한다

## 사전 준비
1. `.env` 파일에 API 키 설정:
```
ANTHROPIC_API_KEY="YOUR_API_KEY_HERE"
```

In [ ]:
# 패키지 설치
%pip install anthropic python-dotenv

In [ ]:
# 환경 설정
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

def llm_call(prompt: str, system: str = None, temperature: float = 0.2) -> str:
    """단일 LLM 호출 헬퍼."""
    params = {
        "model": model,
        "max_tokens": 2000,
        "temperature": temperature,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        params["system"] = system
    response = client.messages.create(**params)
    return response.content[0].text

print("환경 설정 완료")

---
## Exercise 1: Chaining 워크플로 — 번역 파이프라인

### 문제
다음 3단계 체이닝 워크플로를 구현하세요:
1. **Step 1**: 영어 기술 문서를 한국어로 번역
2. **Step 2**: 번역 품질을 평가 (1-10점 + 피드백)
3. **Step 3**: 피드백을 반영하여 번역 개선

각 단계 사이에 **게이트**를 추가하세요:
- Gate 1: 번역 결과가 비어있지 않은지 확인
- Gate 2: 평가 점수가 7점 이상이면 Step 3 생략

### 입력 텍스트
```
The reinforced concrete beam must satisfy both flexural and shear strength requirements 
according to ACI 318-19. The nominal moment capacity shall exceed the factored moment 
demand by at least 20% to ensure adequate safety margin.
```

### 기대 출력
```
[Step 1] 번역 완료 (XXX자)
[Gate 1] 통과
[Step 2] 평가 점수: X/10
[Gate 2] 점수 < 7 → Step 3 실행 / 점수 >= 7 → 완료
[Step 3] 개선된 번역 (해당 시)
```

In [ ]:
# TODO: 3단계 체이닝 워크플로를 구현하세요

import json

input_text = (
    "The reinforced concrete beam must satisfy both flexural and shear strength requirements "
    "according to ACI 318-19. The nominal moment capacity shall exceed the factored moment "
    "demand by at least 20% to ensure adequate safety margin."
)

# Step 1: 번역
# YOUR CODE HERE

# Gate 1: 번역 결과 검증
# YOUR CODE HERE

# Step 2: 평가
# YOUR CODE HERE

# Gate 2: 점수 확인
# YOUR CODE HERE

# Step 3: 개선 (필요 시)
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

import json

input_text = (
    "The reinforced concrete beam must satisfy both flexural and shear strength requirements "
    "according to ACI 318-19. The nominal moment capacity shall exceed the factored moment "
    "demand by at least 20% to ensure adequate safety margin."
)

# Step 1: 번역
translation = llm_call(
    f"다음 영어 기술 문서를 한국어로 번역하라. 번역문만 출력하라:\n\n{input_text}"
)
print(f"[Step 1] 번역 완료 ({len(translation)}자)")
print(f"  {translation[:100]}...\n")

# Gate 1: 번역 결과 검증
if not translation or len(translation.strip()) < 10:
    print("[Gate 1] 실패: 번역 결과가 비어있음")
else:
    print("[Gate 1] 통과")

    # Step 2: 평가
    eval_prompt = (
        f"다음 번역의 품질을 평가하라.\n"
        f"원문: {input_text}\n"
        f"번역: {translation}\n\n"
        f'JSON으로 답하라: {{"score": 1-10, "feedback": "개선사항"}}\nJSON만 출력하라.'
    )
    eval_response = client.messages.create(
        model=model, max_tokens=500, temperature=0.0,
        messages=[
            {"role": "user", "content": eval_prompt},
            {"role": "assistant", "content": "```json\n"}
        ],
        stop_sequences=["```"]
    )
    evaluation = json.loads(eval_response.content[0].text.strip())
    score = evaluation.get("score", 0)
    feedback = evaluation.get("feedback", "")
    print(f"[Step 2] 평가 점수: {score}/10")
    print(f"  피드백: {feedback}\n")

    # Gate 2: 점수 확인
    if score >= 7:
        print(f"[Gate 2] 점수 {score} >= 7 → 완료 (Step 3 생략)")
        final_translation = translation
    else:
        print(f"[Gate 2] 점수 {score} < 7 → Step 3 실행")

        # Step 3: 개선
        improved = llm_call(
            f"다음 번역을 피드백을 반영하여 개선하라.\n"
            f"원문: {input_text}\n"
            f"현재 번역: {translation}\n"
            f"피드백: {feedback}\n\n"
            f"개선된 번역만 출력하라."
        )
        print(f"[Step 3] 개선된 번역:")
        print(f"  {improved}")
        final_translation = improved

    print(f"\n=== 최종 번역 ===")
    print(final_translation)

In [ ]:
# 검증 함수
def verify_exercise_1():
    """Exercise 1 결과를 검증한다."""
    checks = []

    # Check 1: translation 변수 존재
    checks.append("translation" in dir() or "translation" in globals())

    # Check 2: 번역 결과가 한국어를 포함
    if 'final_translation' in globals():
        has_korean = any('가' <= c <= '힣' for c in final_translation)
        checks.append(has_korean)
    else:
        checks.append(False)

    # Check 3: 핵심 용어 포함 여부
    if 'final_translation' in globals():
        has_terms = any(term in final_translation for term in ["철근", "콘크리트", "보", "강도", "모멘트"])
        checks.append(has_terms)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_1()

---
## Exercise 2: Routing 워크플로 — 질문 유형별 분기

### 문제
다음 라우팅 워크플로를 구현하세요:
1. **Router**: 사용자 질문의 유형을 분류 (`calculation`, `explanation`, `code`, `comparison`)
2. **Handlers**: 유형별 전문 시스템 프롬프트를 적용하여 응답

### 전문 핸들러 시스템 프롬프트
- `calculation`: "단계별 계산 과정을 보여주고 결과를 검증하라"
- `explanation`: "초보자도 이해할 수 있게 비유와 예시를 들어 설명하라"
- `code`: "완전히 실행 가능한 Python 코드를 작성하고 주석을 달아라"
- `comparison`: "항목별 비교 표를 만들고 장단점을 정리하라"

### 테스트 질문 (4개 모두 처리)
```python
questions = [
    "500x500mm 기둥의 축력비를 계산해줘. fck=27MPa, Pu=3000kN",
    "프리스트레스 콘크리트란 무엇인가?",
    "RC 보의 휨 강도를 계산하는 Python 함수를 작성해줘",
    "RC 구조와 철골 구조의 장단점을 비교해줘"
]
```

### 기대 출력
```
질문 1 → [Router: calculation] → (계산 과정 포함 응답)
질문 2 → [Router: explanation] → (비유/예시 포함 응답)
질문 3 → [Router: code] → (Python 코드 포함 응답)
질문 4 → [Router: comparison] → (비교 표 포함 응답)
```

In [ ]:
# TODO: 라우팅 워크플로를 구현하세요

questions = [
    "500x500mm 기둥의 축력비를 계산해줘. fck=27MPa, Pu=3000kN",
    "프리스트레스 콘크리트란 무엇인가?",
    "RC 보의 휨 강도를 계산하는 Python 함수를 작성해줘",
    "RC 구조와 철골 구조의 장단점을 비교해줘"
]

# Router 함수 정의
# YOUR CODE HERE

# Handler 시스템 프롬프트 정의
# YOUR CODE HERE

# 라우팅 실행
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

questions = [
    "500x500mm 기둥의 축력비를 계산해줘. fck=27MPa, Pu=3000kN",
    "프리스트레스 콘크리트란 무엇인가?",
    "RC 보의 휨 강도를 계산하는 Python 함수를 작성해줘",
    "RC 구조와 철골 구조의 장단점을 비교해줘"
]

# Router
def route_question(question: str) -> str:
    """질문 유형을 분류한다."""
    prompt = (
        f"다음 질문의 유형을 분류하라. "
        f"반드시 다음 중 하나만 출력하라: calculation, explanation, code, comparison\n\n"
        f"질문: {question}"
    )
    result = llm_call(prompt, temperature=0.0).strip().lower()
    valid_types = ["calculation", "explanation", "code", "comparison"]
    for vt in valid_types:
        if vt in result:
            return vt
    return "explanation"  # 기본값

# Handlers
handler_systems = {
    "calculation": "단계별 계산 과정을 보여주고 결과를 검증하라. 수식과 단위를 명확히 표기하라.",
    "explanation": "초보자도 이해할 수 있게 비유와 예시를 들어 설명하라. 전문 용어는 풀어서 설명하라.",
    "code": "완전히 실행 가능한 Python 코드를 작성하고 상세한 주석을 달아라.",
    "comparison": "항목별 비교 표를 만들고 장단점을 정리하라. 결론을 제시하라."
}

# 라우팅 실행
routing_results = []
for i, q in enumerate(questions, 1):
    q_type = route_question(q)
    system = handler_systems[q_type]
    response = llm_call(q, system=system)
    routing_results.append({"question": q, "type": q_type, "response": response})

    print(f"\n{'='*60}")
    print(f"질문 {i}: {q}")
    print(f"[Router: {q_type}]")
    print(f"\n{response[:300]}..." if len(response) > 300 else f"\n{response}")

In [ ]:
# 검증 함수
def verify_exercise_2():
    """Exercise 2 결과를 검증한다."""
    checks = []

    # Check 1: routing_results 존재하고 4개인지
    if 'routing_results' in globals() and len(routing_results) == 4:
        checks.append(True)
    else:
        checks.append(False)

    # Check 2: 각 결과에 type과 response가 있는지
    if 'routing_results' in globals():
        all_have_keys = all('type' in r and 'response' in r for r in routing_results)
        checks.append(all_have_keys)
    else:
        checks.append(False)

    # Check 3: 라우팅이 올바른지 (첫 번째는 calculation이어야 함)
    if 'routing_results' in globals() and len(routing_results) >= 1:
        checks.append(routing_results[0]['type'] == 'calculation')
    else:
        checks.append(False)

    # Check 4: 모든 응답이 비어있지 않은지
    if 'routing_results' in globals():
        all_have_content = all(len(r['response']) > 50 for r in routing_results)
        checks.append(all_have_content)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_2()

---
## Exercise 3: Parallelization — Sectioning과 Voting

### 문제
다음 두 가지 병렬화 패턴을 구현하세요:

**Part A — Sectioning**: 하나의 건축 프로젝트 설명을 3가지 관점에서 동시에 분석
- 구조적 관점 (structural)
- 경제적 관점 (economic)
- 시공성 관점 (constructability)

**Part B — Voting**: 동일한 구조 안전성 판정을 5회 수행하여 다수결

### 입력 데이터
```
프로젝트: 30층 주거 건물
구조 시스템: RC 전단벽-골조 시스템
콘크리트: fck = 40 MPa (고강도)
특이사항: 지하 3층, 전이보 있음, 내진설계범주 D
```

### 기대 출력
```
=== Sectioning Results ===
[구조적 관점] ...
[경제적 관점] ...
[시공성 관점] ...

=== Voting Results ===
투표: ['OK', 'OK', 'NG', 'OK', 'OK']
판정: OK (신뢰도: 80%)
```

In [ ]:
# TODO: Parallelization (Sectioning + Voting) 구현

import asyncio
from anthropic import AsyncAnthropic

async_client = AsyncAnthropic()

project_desc = (
    "프로젝트: 30층 주거 건물\n"
    "구조 시스템: RC 전단벽-골조 시스템\n"
    "콘크리트: fck = 40 MPa (고강도)\n"
    "특이사항: 지하 3층, 전이보 있음, 내진설계범주 D"
)

# Part A: Sectioning
# YOUR CODE HERE

# Part B: Voting
# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

import asyncio
from anthropic import AsyncAnthropic

async_client = AsyncAnthropic()

project_desc = (
    "프로젝트: 30층 주거 건물\n"
    "구조 시스템: RC 전단벽-골조 시스템\n"
    "콘크리트: fck = 40 MPa (고강도)\n"
    "특이사항: 지하 3층, 전이보 있음, 내진설계범주 D"
)

# === Part A: Sectioning ===
async def analyze_perspective(perspective: str, data: str) -> dict:
    """하나의 관점에서 프로젝트를 분석한다."""
    response = await async_client.messages.create(
        model=model, max_tokens=800, temperature=0.3,
        messages=[{
            "role": "user",
            "content": f"다음 프로젝트를 '{perspective}' 관점에서 분석하라. 핵심 포인트 3가지를 제시하라.\n\n{data}"
        }]
    )
    return {"perspective": perspective, "analysis": response.content[0].text}

async def sectioning_analysis(data: str) -> list[dict]:
    perspectives = ["구조적 관점 (structural)", "경제적 관점 (economic)", "시공성 관점 (constructability)"]
    tasks = [analyze_perspective(p, data) for p in perspectives]
    return await asyncio.gather(*tasks)

# === Part B: Voting ===
async def single_safety_vote(data: str) -> str:
    """단일 안전성 판정."""
    response = await async_client.messages.create(
        model=model, max_tokens=50, temperature=0.5,
        messages=[{
            "role": "user",
            "content": f"다음 프로젝트의 구조적 안전성을 판정하라. 'OK' 또는 'NG'로만 답하라.\n\n{data}"
        }]
    )
    return response.content[0].text.strip()

async def voting_safety(data: str, n_votes: int = 5) -> dict:
    tasks = [single_safety_vote(data) for _ in range(n_votes)]
    votes = await asyncio.gather(*tasks)
    ok_count = sum(1 for v in votes if "OK" in v.upper())
    ng_count = sum(1 for v in votes if "NG" in v.upper())
    total_valid = ok_count + ng_count
    return {
        "votes": votes,
        "verdict": "OK" if ok_count > ng_count else "NG",
        "confidence": max(ok_count, ng_count) / max(total_valid, 1)
    }

# === 실행 ===
print("=== Sectioning Results ===")
section_results = await sectioning_analysis(project_desc)
for r in section_results:
    print(f"\n[{r['perspective']}]")
    print(r['analysis'][:200] + "..." if len(r['analysis']) > 200 else r['analysis'])

print("\n" + "="*60)
print("\n=== Voting Results ===")
vote_result = await voting_safety(project_desc, n_votes=5)
print(f"투표: {vote_result['votes']}")
print(f"판정: {vote_result['verdict']} (신뢰도: {vote_result['confidence']:.0%})")

In [ ]:
# 검증 함수
def verify_exercise_3():
    """Exercise 3 결과를 검증한다."""
    checks = []

    # Check 1: Sectioning 결과가 3개인지
    if 'section_results' in globals() and len(section_results) == 3:
        checks.append(True)
    else:
        checks.append(False)

    # Check 2: Voting 결과에 votes와 verdict가 있는지
    if 'vote_result' in globals():
        checks.append('votes' in vote_result and 'verdict' in vote_result)
    else:
        checks.append(False)

    # Check 3: Voting에 5개 투표가 있는지
    if 'vote_result' in globals() and 'votes' in vote_result:
        checks.append(len(vote_result['votes']) == 5)
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_3()

---
## Exercise 4: 건축공학 응용 — 구조 검토 통합 워크플로

### 문제
Chaining + Routing + Parallelization을 결합하여 구조 검토 시스템을 구축하세요.

**파이프라인:**
1. **Step 1 (Chaining)**: 비정형 입력에서 부재 정보를 JSON으로 파싱
2. **Step 2 (Routing)**: 각 부재의 유형을 분류 (column/beam)
3. **Step 3 (Parallelization)**: 모든 부재를 동시에 전문 검토
4. **Step 4 (Chaining)**: 결과를 종합 보고서로 생성

### 입력
```
1층 구조 부재:
- C1: 500x500mm 기둥, 콘크리트 27MPa, 주근 8-D25, 축력 3000kN
- B1: 350x600mm 보, 콘크리트 24MPa, 인장근 5-D25, 모멘트 380kN·m
- C2: 400x400mm 기둥, 콘크리트 24MPa, 주근 8-D22, 축력 2000kN
```

### 기대 출력
```
[Step 1] 3개 부재 파싱 완료
[Step 2] C1→column, B1→beam, C2→column
[Step 3] 3개 부재 병렬 검토 완료
[Step 4] 종합 보고서 생성 완료
```

In [ ]:
# TODO: Chaining + Routing + Parallelization 통합 워크플로 구현

raw_input = """
1층 구조 부재:
- C1: 500x500mm 기둥, 콘크리트 27MPa, 주근 8-D25, 축력 3000kN
- B1: 350x600mm 보, 콘크리트 24MPa, 인장근 5-D25, 모멘트 380kN·m
- C2: 400x400mm 기둥, 콘크리트 24MPa, 주근 8-D22, 축력 2000kN
"""

# YOUR CODE HERE

In [ ]:
# === 솔루션 ===

import json
import asyncio
from anthropic import AsyncAnthropic

async_client = AsyncAnthropic()

raw_input_text = """
1층 구조 부재:
- C1: 500x500mm 기둥, 콘크리트 27MPa, 주근 8-D25, 축력 3000kN
- B1: 350x600mm 보, 콘크리트 24MPa, 인장근 5-D25, 모멘트 380kN·m
- C2: 400x400mm 기둥, 콘크리트 24MPa, 주근 8-D22, 축력 2000kN
"""

# Step 1: 파싱 (Chaining)
print("[Step 1] 부재 정보 파싱 중...")
parse_response = client.messages.create(
    model=model, max_tokens=1000, temperature=0.0,
    messages=[
        {"role": "user", "content": (
            f"다음 텍스트에서 구조 부재 정보를 추출하여 JSON 배열로 반환하라. "
            f"각 부재: name, type(column/beam), section, fck_MPa, rebar, loads\n\n{raw_input_text}"
        )},
        {"role": "assistant", "content": "```json\n"}
    ],
    stop_sequences=["```"]
)
members = json.loads(parse_response.content[0].text.strip())
print(f"  → {len(members)}개 부재 파싱 완료")

# Step 2: 라우팅 (각 부재 유형 확인)
print("[Step 2] 부재 유형 분류...")
for m in members:
    m_type = m.get("type", "unknown")
    print(f"  {m['name']} → {m_type}")

# Step 3: 병렬 검토 (Parallelization)
print("[Step 3] 병렬 검토 중...")

system_map = {
    "column": "RC 기둥 전문가. KDS 14 20 기준 축력비, 철근비를 검토하라.",
    "beam": "RC 보 전문가. KDS 14 20 기준 휨 강도, 전단을 검토하라."
}

async def review_one_member(member: dict) -> dict:
    m_type = member.get("type", "column")
    sys = system_map.get(m_type, "구조 부재를 검토하라.")
    resp = await async_client.messages.create(
        model=model, max_tokens=1000, temperature=0.2,
        system=sys,
        messages=[{"role": "user", "content": f"검토: {json.dumps(member, ensure_ascii=False)}"}]
    )
    return {"name": member["name"], "type": m_type, "review": resp.content[0].text}

reviews = await asyncio.gather(*[review_one_member(m) for m in members])
print(f"  → {len(reviews)}개 부재 검토 완료")

# Step 4: 종합 보고서 (Chaining)
print("[Step 4] 종합 보고서 생성 중...")
reviews_text = "\n\n".join(f"### {r['name']} ({r['type']})\n{r['review']}" for r in reviews)
final_report = llm_call(
    f"다음 개별 검토 결과를 종합 보고서로 정리하라:\n\n{reviews_text}"
)
print("  → 보고서 생성 완료")
print("\n" + "="*60)
print(final_report)

In [ ]:
# 검증 함수
def verify_exercise_4():
    """Exercise 4 결과를 검증한다."""
    checks = []

    # Check 1: 부재가 3개 파싱되었는지
    if 'members' in globals():
        checks.append(len(members) == 3)
    else:
        checks.append(False)

    # Check 2: 검토 결과가 3개인지
    if 'reviews' in globals():
        checks.append(len(reviews) == 3)
    else:
        checks.append(False)

    # Check 3: 최종 보고서가 존재하고 충분한 길이인지
    if 'final_report' in globals():
        checks.append(len(final_report) > 100)
    else:
        checks.append(False)

    # Check 4: 라우팅이 올바른지 (C1=column, B1=beam)
    if 'reviews' in globals() and len(reviews) >= 2:
        types = {r['name']: r['type'] for r in reviews}
        checks.append(types.get('C1') == 'column' and types.get('B1') == 'beam')
    else:
        checks.append(False)

    passed = sum(checks)
    total = len(checks)
    print(f"검증 결과: {passed}/{total} 통과")
    for i, c in enumerate(checks, 1):
        print(f"  Check {i}: {'PASS' if c else 'FAIL'}")
    return passed == total

verify_exercise_4()